# NB1 — Baselines Count et TF-IDF avec tuning intégré

Ce notebook regroupe les baselines classiques `P01` à `P05`. Il conserve l’évaluation initiale sans optimisation, puis ajoute une phase de tuning accéléré des classifieurs.

Ce notebook conserve la **phase baseline sans optimisation**, puis ajoute une **phase d’optimisation accélérée**. 
Le principe retenu est le suivant : pour chaque pipeline, on ajuste d’abord le **préprocesseur / vectoriseur une seule fois** sur un sous-ensemble d’apprentissage, puis on teste plusieurs réglages du **classifieur uniquement** sur les mêmes données déjà transformées. Cela réduit fortement le temps d’exécution.

Cette stratégie est très pratique pour explorer rapidement des réglages d’algorithmes, mais il faut bien comprendre qu’elle constitue une **optimisation accélérée**, plus pragmatique qu’une recherche entièrement relancée sur tout le pipeline à chaque itération.


In [1]:
# Pour un run sur Colab

'''
from google.colab import drive
drive.mount('/content/drive')

PROJECT_ROOT = "/content/drive/MyDrive/Disaster-Tweets-NLP"
MODELS_DIR = f"{PROJECT_ROOT}/notebooks/models_training"

%cd "{MODELS_DIR}"

import sys
if MODELS_DIR not in sys.path:
    sys.path.append(MODELS_DIR)

print("Projet :", PROJECT_ROOT)
print("Dossier courant :", MODELS_DIR)
'''


'\nfrom google.colab import drive\ndrive.mount(\'/content/drive\')\n\nPROJECT_ROOT = "/content/drive/MyDrive/Disaster-Tweets-NLP"\nMODELS_DIR = f"{PROJECT_ROOT}/notebooks/models_training"\n\n%cd "{MODELS_DIR}"\n\nimport sys\nif MODELS_DIR not in sys.path:\n    sys.path.append(MODELS_DIR)\n\nprint("Projet :", PROJECT_ROOT)\nprint("Dossier courant :", MODELS_DIR)\n'

In [2]:
# Installation éventuelle (décommente si nécessaire)
# !pip install pandas numpy scikit-learn scipy matplotlib gensim sentence-transformers openpyxl mlflow

import json
from collections import OrderedDict
from pathlib import Path

import pandas as pd
from mlflow_utils import (
    fit_evaluate_and_log_sklearn_pipeline,
    setup_mlflow_tracking,
)
from nlp_disaster_utils import (
    load_train_test_xy,
    round_results,
    save_results_bundle,
    seed_everything,
    stratified_validation_split,
)
from pipeline_tuning_utils import (
    compare_baseline_vs_tuned,
    evaluate_refit_outputs,
    fit_transform_preprocessor_once,
    log_tuning_run_to_mlflow,
    safe_scores,
    split_pipeline_preprocessor_estimator,
    tune_classifier_on_fixed_features,
)
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn.linear_model import LogisticRegression, SGDClassifier
from sklearn.naive_bayes import MultinomialNB
from sklearn.pipeline import Pipeline
from sklearn.svm import LinearSVC


seed_everything(42)


C:\Users\DELL\AppData\Local\Programs\Python\Python311\Lib\site-packages\pydantic\_internal\_fields.py:161: UserWarning: Field "model_name" has conflict with protected namespace "model_".

You may be able to resolve this warning by setting `model_config['protected_namespaces'] = ()`.
  warnings.warn(


In [3]:
DATA_DIR = "../../data/processed_data"
TRAIN_PATH = f"{DATA_DIR}/train.csv"
TEST_PATH = f"{DATA_DIR}/test.csv"

TEXT_COL = "text"
LABEL_COL = "target"
USE_AUX_TEXT_COLUMNS = False
LOWERCASE_TEXT = False
RANDOM_STATE = 42

OUTPUT_STEM = "NB1_count_tfidf_baselines"
RESULTS_DIR = "../../outputs/NB1"
TUNING_OUTPUT_DIR = Path(RESULTS_DIR) / "tuning"
TUNING_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# Configuration MLflow
MLFLOW_EXPERIMENT_NAME = "DT_NB1_count_tfidf_baselines"
MLFLOW_TRACKING_URI = Path("../../outputs/mlruns").resolve().as_uri()
MLFLOW_LOG_MODEL = False
USE_MLFLOW = True

tracking_uri = setup_mlflow_tracking(
    experiment_name=MLFLOW_EXPERIMENT_NAME,
    tracking_uri=MLFLOW_TRACKING_URI,
)
print("MLflow tracking URI :", tracking_uri)
print("MLflow experiment   :", MLFLOW_EXPERIMENT_NAME)

# Paramètres de tuning accéléré
VAL_SIZE_FOR_TUNING = 0.15
PRIMARY_TUNING_METRIC = "f1_pos"


MLflow tracking URI : file:///C:/Users/DELL/Documents/Classes/ISE2/ISE2_2026/SEM2/ML2/Projet/Disaster-Tweets-NLP/outputs/mlruns
MLflow experiment   : DT_NB1_count_tfidf_baselines


C:\Users\DELL\AppData\Local\Programs\Python\Python311\Lib\site-packages\mlflow\tracking\_tracking_service\utils.py:184: FutureWarning: The filesystem tracking backend (e.g., './mlruns') is deprecated as of February 2026. Consider transitioning to a database backend (e.g., 'sqlite:///mlflow.db') to take advantage of the latest MLflow features. See https://mlflow.org/docs/latest/self-hosting/migrate-from-file-store for migration guidance.
  return FileStore(store_uri, store_uri)


In [4]:
df_train, X_train, y_train, df_test, X_test, y_test = load_train_test_xy(
    train_path=TRAIN_PATH,
    test_path=TEST_PATH,
    text_col=TEXT_COL,
    label_col=LABEL_COL,
    use_extra_cols=USE_AUX_TEXT_COLUMNS,
    lowercase=LOWERCASE_TEXT,
)

print("Taille train :", len(X_train))
print("Taille test  :", len(X_test))
print("\nDistribution des classes - train :")
print(y_train.value_counts(normalize=True).sort_index())
print("\nDistribution des classes - test :")
print(y_test.value_counts(normalize=True).sort_index())


Taille train : 9096
Taille test  : 2274

Distribution des classes - train :
target
0    0.814094
1    0.185906
Name: proportion, dtype: float64

Distribution des classes - test :
target
0    0.813984
1    0.186016
Name: proportion, dtype: float64


In [5]:
pipelines = OrderedDict({
    "P01_Count_MultinomialNB": Pipeline([
        ("vect", CountVectorizer(ngram_range=(1, 1), min_df=2)),
        ("clf", MultinomialNB(alpha=0.5)),
    ]),
    "P02_TFIDF_Unigram_LogReg": Pipeline([
        ("vect", TfidfVectorizer(ngram_range=(1, 1), min_df=2, max_df=0.95)),
        ("clf", LogisticRegression(max_iter=2000, C=1.0, class_weight=None)),
    ]),
    "P03_TFIDF_UniBi_LogReg": Pipeline([
        ("vect", TfidfVectorizer(ngram_range=(1, 2), min_df=2, max_df=0.95)),
        ("clf", LogisticRegression(max_iter=2500, C=1.0, class_weight=None)),
    ]),
    "P04_TFIDF_UniBi_LinearSVC": Pipeline([
        ("vect", TfidfVectorizer(ngram_range=(1, 2), min_df=2, max_df=0.95)),
        ("clf", LinearSVC(C=1.0)),
    ]),
    "P05_TFIDF_UniBi_SGDLog": Pipeline([
        ("vect", TfidfVectorizer(ngram_range=(1, 2), min_df=2, max_df=0.95)),
        ("clf", SGDClassifier(loss="log_loss", penalty="l2", alpha=1e-5, max_iter=3000, random_state=42)),
    ]),
})


## Phase 1 — Baselines sans optimisation

Cette première phase reproduit le benchmark initial : chaque pipeline est exécuté tel quel, avec ses paramètres de départ.

In [6]:
resultats = []
baseline_failures = []

for nom_pipeline, pipeline in pipelines.items():
    print(f"Entraînement baseline -> {nom_pipeline}")
    display(pipeline)
    print("-" * 80)

    try:
        metrics = fit_evaluate_and_log_sklearn_pipeline(
            name=nom_pipeline,
            estimator=pipeline,
            X_train=X_train,
            X_test=X_test,
            y_train=y_train,
            y_test=y_test,
            notebook_name="COUNT",
            family_name="count_tfidf_baselines",
            output_dir=RESULTS_DIR,
            log_model=MLFLOW_LOG_MODEL,
        )
        resultats.append(metrics)
    except Exception as exc:
        baseline_failures.append({"pipeline": nom_pipeline, "error": str(exc)})
        print(f"Échec baseline pour {nom_pipeline} : {exc}")

baseline_df = round_results(pd.DataFrame(resultats))
display(baseline_df)

if baseline_failures:
    print("\nPipelines baseline en échec :")
    display(pd.DataFrame(baseline_failures))


Entraînement baseline -> P01_Count_MultinomialNB


Pipeline(steps=[('vect', CountVectorizer(min_df=2)),
                ('clf', MultinomialNB(alpha=0.5))])

--------------------------------------------------------------------------------
Entraînement baseline -> P02_TFIDF_Unigram_LogReg


Pipeline(steps=[('vect', TfidfVectorizer(max_df=0.95, min_df=2)),
                ('clf', LogisticRegression(max_iter=2000))])

--------------------------------------------------------------------------------
Entraînement baseline -> P03_TFIDF_UniBi_LogReg


...

Pipeline(steps=[('vect',
                 TfidfVectorizer(max_df=0.95, min_df=2, ngram_range=(1, 2))),
                ('clf', LinearSVC())])

--------------------------------------------------------------------------------
Entraînement baseline -> P05_TFIDF_UniBi_SGDLog


Pipeline(steps=[('vect',
                 TfidfVectorizer(max_df=0.95, min_df=2, ngram_range=(1, 2))),
                ('clf',
                 SGDClassifier(alpha=1e-05, loss='log_loss', max_iter=3000,
                               random_state=42))])

--------------------------------------------------------------------------------


pipeline,P01_Count_MultinomialNB,P02_TFIDF_Unigram_LogReg,P03_TFIDF_UniBi_LogReg,P04_TFIDF_UniBi_LinearSVC,P05_TFIDF_UniBi_SGDLog
train_accuracy,0.9291,0.9037,0.8986,0.9956,0.9963
train_precision_macro,0.8767,0.9319,0.9376,0.9957,0.9959
train_recall_macro,0.8958,0.7476,0.7301,0.9898,0.9918
train_f1_macro,0.8858,0.8011,0.7851,0.9927,0.9938
train_precision_weighted,0.9312,0.9100,0.9080,0.9956,0.9963
train_recall_weighted,0.9291,0.9037,0.8986,0.9956,0.9963
train_f1_weighted,0.9300,0.8909,0.8832,0.9956,0.9963
train_precision_class_0,0.9635,0.8970,0.8904,0.9956,0.9965
train_recall_class_0,0.9488,0.9961,0.9984,0.9991,0.9989
train_f1_class_0,0.9561,0.9439,0.9413,0.9973,0.9977


In [7]:
save_results_bundle(pd.DataFrame(resultats), output_dir=RESULTS_DIR, stem=OUTPUT_STEM)
print(f"Fichiers CSV/XLSX baseline enregistrés dans {RESULTS_DIR}")


Fichiers CSV/XLSX baseline enregistrés dans ../../outputs/NB1


## Phase 2 — Tuning accéléré avec vectorisation unique par pipeline

Ici, pour chaque pipeline, on sépare le **préprocesseur** du **classifieur**. On ajuste le préprocesseur **une seule fois** sur un sous-ensemble d’apprentissage, puis on teste différentes combinaisons d’hyperparamètres du classifieur sur les mêmes données déjà vectorisées. Enfin, on réajuste le meilleur classifieur sur tout le train transformé une seule fois et on l’évalue sur train et test.

In [8]:
X_fit, X_val, y_fit, y_val = stratified_validation_split(
    X_train,
    y_train,
    val_size=VAL_SIZE_FOR_TUNING,
    random_state=RANDOM_STATE,
)

print("Taille tuning-fit :", len(X_fit))
print("Taille tuning-val :", len(X_val))


Taille tuning-fit : 7731
Taille tuning-val : 1365


In [9]:
classifier_param_grids = {
    "P01_Count_MultinomialNB": {
        "alpha": [0.1, 0.5, 1.0, 2.0],
    },
    "P02_TFIDF_Unigram_LogReg": {
        "C": [0.25, 0.5, 1.0, 2.0, 4.0],
        "class_weight": [None, "balanced"],
    },
    "P03_TFIDF_UniBi_LogReg": {
        "C": [0.25, 0.5, 1.0, 2.0, 4.0],
        "class_weight": [None, "balanced"],
    },
    "P04_TFIDF_UniBi_LinearSVC": {
        "C": [0.25, 0.5, 1.0, 2.0, 4.0],
        "class_weight": [None, "balanced"],
    },
    "P05_TFIDF_UniBi_SGDLog": {
        "alpha": [1e-6, 1e-5, 1e-4],
        "penalty": ["l2", "elasticnet"],
        "class_weight": [None, "balanced"],
    },
}


In [10]:
tuning_rows = []
tuning_failures = []
tuned_metrics_rows = []

for nom_pipeline, pipeline in pipelines.items():
    print("=" * 100)
    print(f"Tuning accéléré -> {nom_pipeline}")

    param_grid = classifier_param_grids.get(nom_pipeline)
    if param_grid is None:
        tuning_failures.append({"pipeline": nom_pipeline, "error": "Grille d'hyperparamètres absente"})
        print("Aucune grille trouvée.")
        continue

    try:
        preprocessor, clf_name, base_estimator = split_pipeline_preprocessor_estimator(pipeline)

        transformed = fit_transform_preprocessor_once(
            preprocessor=preprocessor,
            X_fit=X_fit,
            y_fit=y_fit,
            X_val=X_val,
        )

        best_params, tuning_results_df = tune_classifier_on_fixed_features(
            base_estimator=base_estimator,
            param_grid=param_grid,
            X_fit=transformed["X_fit_transformed"],
            y_fit=y_fit,
            X_val=transformed["X_val_transformed"],
            y_val=y_val,
            primary_metric=PRIMARY_TUNING_METRIC,
        )

        tuning_results_df.insert(0, "pipeline", nom_pipeline)
        tuning_results_df.insert(1, "classifier_name", clf_name)

        best_preprocessor_full, _, best_estimator_template = split_pipeline_preprocessor_estimator(pipeline)
        best_preprocessor_full.fit(X_train, y_train)
        X_train_vec = best_preprocessor_full.transform(X_train)
        X_test_vec = best_preprocessor_full.transform(X_test)

        best_estimator = base_estimator.set_params(**best_params)
        best_estimator.fit(X_train_vec, y_train)

        train_pred = best_estimator.predict(X_train_vec)
        test_pred = best_estimator.predict(X_test_vec)
        train_score = safe_scores(best_estimator, X_train_vec)
        test_score = safe_scores(best_estimator, X_test_vec)

        final_metrics = evaluate_refit_outputs(
            pipeline_name=nom_pipeline,
            y_train=y_train,
            y_test=y_test,
            train_pred=train_pred,
            test_pred=test_pred,
            train_score=train_score,
            test_score=test_score,
        )
        final_metrics["best_params"] = json.dumps(best_params, ensure_ascii=False)
        final_metrics["best_val_primary_score"] = float(tuning_results_df.iloc[0]["primary_score"])
        final_metrics["best_val_f1_class_1"] = float(tuning_results_df.iloc[0]["val_f1_class_1"])
        final_metrics["best_val_recall_class_1"] = float(tuning_results_df.iloc[0]["val_recall_class_1"])
        final_metrics["best_val_f1_macro"] = float(tuning_results_df.iloc[0]["val_f1_macro"])
        final_metrics["best_val_balanced_accuracy"] = float(tuning_results_df.iloc[0]["val_balanced_accuracy"])

        tuning_rows.append(tuning_results_df.iloc[0].to_dict() | {
            "pipeline": nom_pipeline,
            "best_params": json.dumps(best_params, ensure_ascii=False),
        })
        tuned_metrics_rows.append(final_metrics)

        tuning_results_path = TUNING_OUTPUT_DIR / f"{nom_pipeline}_tuning_validation_results.csv"
        tuning_results_df.to_csv(tuning_results_path, index=False)

        if USE_MLFLOW:
            log_tuning_run_to_mlflow(
                run_name=nom_pipeline,
                notebook_name="COUNT",
                family_name="count_tfidf_baselines",
                best_params=best_params,
                tuning_results_df=tuning_results_df,
                final_metrics=final_metrics,
                output_dir=TUNING_OUTPUT_DIR,
            )

        print("Meilleurs paramètres :", best_params)
        print("Meilleur score de validation :", tuning_results_df.iloc[0]["primary_score"])

    except Exception as exc:
        tuning_failures.append({"pipeline": nom_pipeline, "error": str(exc)})
        print(f"Échec tuning pour {nom_pipeline} : {exc}")


Tuning accéléré -> P01_Count_MultinomialNB
Meilleurs paramètres : {'alpha': 0.1}
Meilleur score de validation : 0.6851485148514852
Tuning accéléré -> P02_TFIDF_Unigram_LogReg
Meilleurs paramètres : {'C': 2.0, 'class_weight': 'balanced'}
Meilleur score de validation : 0.6689303904923599
Tuning accéléré -> P03_TFIDF_UniBi_LogReg
Meilleurs paramètres : {'C': 2.0, 'class_weight': 'balanced'}
Meilleur score de validation : 0.6858168761220825
Tuning accéléré -> P04_TFIDF_UniBi_LinearSVC
Meilleurs paramètres : {'C': 2.0, 'class_weight': None}
Meilleur score de validation : 0.7115789473684211
Tuning accéléré -> P05_TFIDF_UniBi_SGDLog
Meilleurs paramètres : {'alpha': 1e-05, 'class_weight': 'balanced', 'penalty': 'l2'}
Meilleur score de validation : 0.6977611940298507


In [11]:
tuning_best_df = round_results(pd.DataFrame(tuning_rows))
display(tuning_best_df)

tuned_results_df = round_results(pd.DataFrame(tuned_metrics_rows))
display(tuned_results_df)

if tuning_failures:
    print("\nPipelines tuning en échec :")
    display(pd.DataFrame(tuning_failures))


pipeline,P01_Count_MultinomialNB,P02_TFIDF_Unigram_LogReg,P03_TFIDF_UniBi_LogReg,P04_TFIDF_UniBi_LinearSVC,P05_TFIDF_UniBi_SGDLog
classifier_name,clf,clf,clf,clf,clf
params,"{""alpha"": 0.1}","{""C"": 2.0, ""class_weight"": ""balanced""}","{""C"": 2.0, ""class_weight"": ""balanced""}","{""C"": 2.0, ""class_weight"": null}","{""alpha"": 1e-05, ""class_weight"": ""balanced"", ""penalty"": ""l2""}"
primary_metric,f1_pos,f1_pos,f1_pos,f1_pos,f1_pos
primary_score,0.6851,0.6689,0.6858,0.7116,0.6978
val_accuracy,0.8835,0.8571,0.8718,0.8996,0.8813
val_precision_macro,0.8083,0.7664,0.7855,0.8452,0.8006
val_recall_macro,0.8054,0.8257,0.8256,0.8093,0.8254
val_f1_macro,0.8068,0.7889,0.8026,0.8254,0.8120
val_precision_weighted,0.8830,0.8783,0.8829,0.8957,0.8870
val_recall_weighted,0.8835,0.8571,0.8718,0.8996,0.8813


pipeline,P01_Count_MultinomialNB,P02_TFIDF_Unigram_LogReg,P03_TFIDF_UniBi_LogReg,P04_TFIDF_UniBi_LinearSVC,P05_TFIDF_UniBi_SGDLog
train_accuracy,0.9457,0.9439,0.9686,0.9980,0.9942
train_precision_macro,0.9056,0.8875,0.9299,0.9976,0.9848
train_recall_macro,0.9180,0.9521,0.9761,0.9958,0.9964
train_f1_macro,0.9117,0.9147,0.9508,0.9967,0.9905
train_precision_weighted,0.9467,0.9529,0.9722,0.9980,0.9944
train_recall_weighted,0.9457,0.9439,0.9686,0.9980,0.9942
train_f1_weighted,0.9461,0.9461,0.9694,0.9980,0.9942
train_precision_class_0,0.9710,0.9916,0.9972,0.9982,1.0000
train_recall_class_0,0.9621,0.9391,0.9641,0.9993,0.9928
train_f1_class_0,0.9665,0.9646,0.9804,0.9988,0.9964


In [12]:
comparison_df = compare_baseline_vs_tuned(
    baseline_df=pd.DataFrame(resultats),
    tuned_df=pd.DataFrame(tuned_metrics_rows),
)
display(round_results(comparison_df))


,baseline_pipeline,baseline_train_accuracy,baseline_train_precision_macro,baseline_train_recall_macro,baseline_train_f1_macro,baseline_train_precision_weighted,baseline_train_recall_weighted,baseline_train_f1_weighted,baseline_train_precision_class_0,baseline_train_recall_class_0,baseline_train_f1_class_0,baseline_train_support_class_0,baseline_train_precision_class_1,baseline_train_recall_class_1,baseline_train_f1_class_1,baseline_train_support_class_1,baseline_train_balanced_accuracy,baseline_train_roc_auc,baseline_train_pr_auc,baseline_test_accuracy,baseline_test_precision_macro,baseline_test_recall_macro,baseline_test_f1_macro,baseline_test_precision_weighted,baseline_test_recall_weighted,baseline_test_f1_weighted,baseline_test_precision_class_0,baseline_test_recall_class_0,baseline_test_f1_class_0,baseline_test_support_class_0,baseline_test_precision_class_1,baseline_test_recall_class_1,baseline_test_f1_class_1,baseline_test_support_class_1,baseline_test_balanced_accuracy,baseline_test_roc_auc,baseline_test_pr_auc,tuned_pipeline,tuned_train_accuracy,tuned_train_precision_macro,tuned_train_recall_macro,tuned_train_f1_macro,tuned_train_precision_weighted,tuned_train_recall_weighted,tuned_train_f1_weighted,tuned_train_precision_class_0,tuned_train_recall_class_0,tuned_train_f1_class_0,tuned_train_support_class_0,tuned_train_precision_class_1,tuned_train_recall_class_1,tuned_train_f1_class_1,tuned_train_support_class_1,tuned_train_balanced_accuracy,tuned_train_roc_auc,tuned_train_pr_auc,tuned_test_accuracy,tuned_test_precision_macro,tuned_test_recall_macro,tuned_test_f1_macro,tuned_test_precision_weighted,tuned_test_recall_weighted,tuned_test_f1_weighted,tuned_test_precision_class_0,tuned_test_recall_class_0,tuned_test_f1_class_0,tuned_test_support_class_0,tuned_test_precision_class_1,tuned_test_recall_class_1,tuned_test_f1_class_1,tuned_test_support_class_1,tuned_test_balanced_accuracy,tuned_test_roc_auc,tuned_test_pr_auc,tuned_best_params,tuned_best_val_primary_score,tuned_best_val_f1_class_1,tuned_best_val_recall_class_1,tuned_best_val_f1_macro,tuned_best_val_balanced_accuracy,delta_test_f1_class_1,delta_test_recall_class_1,delta_test_f1_macro,delta_test_balanced_accuracy
0,P01_Count_MultinomialNB,0.9291,0.8767,0.8958,0.8858,0.9312,0.9291,0.9300,0.9635,0.9488,0.9561,7405.0000,0.7899,0.8427,0.8155,1691.0000,0.8958,0.9610,0.8981,0.8799,0.7991,0.8177,0.8078,0.8841,0.8799,0.8818,0.9345,0.9168,0.9256,1851.0000,0.6638,0.7187,0.6901,423.0000,0.8177,0.9054,0.7616,P01_Count_MultinomialNB,0.9457,0.9056,0.9180,0.9117,0.9467,0.9457,0.9461,0.9710,0.9621,0.9665,7405.0000,0.8403,0.8740,0.8568,1691.0000,0.9180,0.9774,0.9306,0.8821,0.8039,0.8127,0.8082,0.8839,0.8821,0.8830,0.9313,0.9233,0.9273,1851.0000,0.6765,0.7021,0.6891,423.0000,0.8127,0.9034,0.7584,"{""alpha"": 0.1}",0.6851,0.6851,0.6811,0.8068,0.8054,-0.0010,-0.0165,0.0004,-0.0050
1,P02_TFIDF_Unigram_LogReg,0.9037,0.9319,0.7476,0.8011,0.9100,0.9037,0.8909,0.8970,0.9961,0.9439,7405.0000,0.9668,0.4991,0.6583,1691.0000,0.7476,0.9638,0.8970,0.8738,0.8601,0.6890,0.7324,0.8705,0.8738,0.8545,0.8767,0.9833,0.9269,1851.0000,0.8434,0.3948,0.5378,423.0000,0.6890,0.9133,0.7502,P02_TFIDF_Unigram_LogReg,0.9439,0.8875,0.9521,0.9147,0.9529,0.9439,0.9461,0.9916,0.9391,0.9646,7405.0000,0.7835,0.9651,0.8649,1691.0000,0.9521,0.9877,0.9463,0.8615,0.7725,0.8365,0.7965,0.8838,0.8615,0.8687,0.9496,0.8763,0.9115,1851.0000,0.5954,0.7967,0.6815,423.0000,0.8365,0.9174,0.7462,"{""C"": 2.0, ""class_weight"": ""balanced""}",0.6689,0.6689,0.7756,0.7889,0.8257,0.1437,0.4019,0.0641,0.1475
2,P03_TFIDF_UniBi_LogReg,0.8986,0.9376,0.7301,0.7851,0.9080,0.8986,0.8832,0.8904,0.9984,0.9413,7405.0000,0.9849,0.4619,0.6288,1691.0000,0.7301,0.9768,0.9313,0.8698,0.8789,0.6665,0.7101,0.8723,0.8698,0.8452,0.8683,0.9903,0.9253,1851.0000,0.8896,0.3428,0.4949,423.0000,0.6665,0.9157,0.7677,P03_TFIDF_UniBi_LogReg,0.9686,0.9299,0.9761,0.9508,0.9722,0.9686,0.9694,0.9972,0.9641,0.9804,7405.0000,0.8627,0.9882,0.9212,1691.0000,0.9761,0.9956,

In [13]:
pd.DataFrame(tuning_rows).to_csv(TUNING_OUTPUT_DIR / f"{OUTPUT_STEM}_tuning_resume.csv", index=False)
pd.DataFrame(tuned_metrics_rows).to_csv(TUNING_OUTPUT_DIR / f"{OUTPUT_STEM}_tuned_final_results.csv", index=False)
pd.DataFrame(tuning_failures).to_csv(TUNING_OUTPUT_DIR / f"{OUTPUT_STEM}_tuning_failures.csv", index=False)

comparison_df.to_csv(TUNING_OUTPUT_DIR / f"{OUTPUT_STEM}_baseline_vs_tuned.csv", index=False)

print("Exports tuning enregistrés dans :", TUNING_OUTPUT_DIR)


Exports tuning enregistrés dans : ..\..\outputs\NB1\tuning


Le notebook contient désormais les deux temps de travail : un benchmark initial sans optimisation, puis une optimisation accélérée centrée sur les algorithmes.